# Create Azure Container Apps GPU Job for GPT SFT

This notebook creates or updates a **separate ACA Job** for supervised fine-tuning using the existing Container Apps Environment that was already created for pretraining.

It does **not** recreate the ACA environment. It only:

- loads config from `aca_sft/.env`
- authenticates to Azure
- verifies the existing ACA environment
- creates or updates `gpt-sft-job`
- starts a manual execution

Expected image:

- `docker.io/deril2605/gpt-sft:latest`


In [ ]:
%pip install --quiet azure-identity azure-mgmt-resource azure-mgmt-appcontainers requests python-dotenv

In [ ]:
import json
import os
from pathlib import Path

import requests
from dotenv import load_dotenv
from azure.identity import InteractiveBrowserCredential
from azure.mgmt.appcontainers import ContainerAppsAPIClient
from azure.mgmt.resource import ResourceManagementClient

load_dotenv()
load_dotenv(Path.cwd() / '.env')
load_dotenv(Path.cwd() / 'aca_sft' / '.env')

SUBSCRIPTION_ID = os.environ['AZURE_SUBSCRIPTION_ID']
AZURE_TENANT_ID = os.environ['AZURE_TENANT_ID']
RESOURCE_GROUP = os.environ['AZURE_RESOURCE_GROUP']
LOCATION = os.getenv('AZURE_LOCATION', 'eastus')

ENV_NAME = os.getenv('AZURE_CONTAINERAPPS_ENV_NAME', 'cae-gpt-pretraining-gpu')
JOB_NAME = os.getenv('AZURE_CONTAINERAPP_JOB_NAME', 'gpt-sft-job')
GPU_PROFILE_NAME = os.getenv('AZURE_GPU_PROFILE_NAME', 'Consumption-GPU-NC24-A100')

TRAINING_IMAGE = os.getenv('ACA_JOB_IMAGE', 'docker.io/deril2605/gpt-sft:latest')
CPU = float(os.getenv('ACA_JOB_CPU', '24'))
MEMORY = os.getenv('ACA_JOB_MEMORY', '220Gi')
REPLICA_TIMEOUT_SECONDS = int(os.getenv('ACA_JOB_REPLICA_TIMEOUT_SECONDS', '86400'))
REPLICA_RETRY_LIMIT = int(os.getenv('ACA_JOB_REPLICA_RETRY_LIMIT', '0'))
PARALLELISM = int(os.getenv('ACA_JOB_PARALLELISM', '1'))
REPLICA_COMPLETION_COUNT = int(os.getenv('ACA_JOB_REPLICA_COMPLETION_COUNT', '1'))

BLOB_CONNECTION_STRING = os.environ['AZURE_STORAGE_CONNECTION_STRING']
BLOB_CONTAINER = os.environ['AZURE_BLOB_CONTAINER']
PRETRAINED_MODEL_BLOB_PATH = os.environ['PRETRAINED_MODEL_BLOB_PATH']
LOCAL_PRETRAINED_CHECKPOINT_PATH = os.getenv('LOCAL_PRETRAINED_CHECKPOINT_PATH', '/app/model_inputs/pretrain_final.pth')

ARM_API_VERSION = '2025-01-01'

print(json.dumps({
    'subscription_id': SUBSCRIPTION_ID,
    'tenant_id': AZURE_TENANT_ID,
    'resource_group': RESOURCE_GROUP,
    'location': LOCATION,
    'environment_name': ENV_NAME,
    'job_name': JOB_NAME,
    'gpu_profile_name': GPU_PROFILE_NAME,
    'training_image': TRAINING_IMAGE,
    'cpu': CPU,
    'memory': MEMORY,
    'pretrained_model_blob_path': PRETRAINED_MODEL_BLOB_PATH,
}, indent=2))

In [ ]:
credential = InteractiveBrowserCredential(tenant_id=AZURE_TENANT_ID)
resource_client = ResourceManagementClient(credential, SUBSCRIPTION_ID)
app_client = ContainerAppsAPIClient(credential=credential, subscription_id=SUBSCRIPTION_ID)

resource_group = resource_client.resource_groups.get(RESOURCE_GROUP)
print(f'Resource group found: {resource_group.name}')

In [ ]:
environment = app_client.managed_environments.get(RESOURCE_GROUP, ENV_NAME)
ENV_RESOURCE_ID = environment.id

print('Existing ACA environment found:')
print(json.dumps({
    'name': environment.name,
    'id': ENV_RESOURCE_ID,
    'location': environment.location,
}, indent=2))

In [ ]:
job_payload = {
    'location': LOCATION,
    'properties': {
        'environmentId': ENV_RESOURCE_ID,
        'workloadProfileName': GPU_PROFILE_NAME,
        'configuration': {
            'triggerType': 'Manual',
            'replicaTimeout': REPLICA_TIMEOUT_SECONDS,
            'replicaRetryLimit': REPLICA_RETRY_LIMIT,
            'manualTriggerConfig': {
                'parallelism': PARALLELISM,
                'replicaCompletionCount': REPLICA_COMPLETION_COUNT,
            },
            'secrets': [
                {
                    'name': 'azure-storage-connection-string',
                    'value': BLOB_CONNECTION_STRING,
                }
            ],
        },
        'template': {
            'containers': [
                {
                    'name': 'trainer',
                    'image': TRAINING_IMAGE,
                    'resources': {
                        'cpu': CPU,
                        'memory': MEMORY,
                    },
                    'env': [
                        {'name': 'AZURE_BLOB_CONTAINER', 'value': BLOB_CONTAINER},
                        {'name': 'AZURE_STORAGE_CONNECTION_STRING', 'secretRef': 'azure-storage-connection-string'},
                        {'name': 'PRETRAINED_MODEL_BLOB_PATH', 'value': PRETRAINED_MODEL_BLOB_PATH},
                        {'name': 'LOCAL_PRETRAINED_CHECKPOINT_PATH', 'value': LOCAL_PRETRAINED_CHECKPOINT_PATH},
                        {'name': 'PYTHONUNBUFFERED', 'value': '1'},
                    ],
                }
            ]
        },
    },
}

print(json.dumps(job_payload, indent=2)[:5000])

In [ ]:
job_url = (
    f'https://management.azure.com/subscriptions/{SUBSCRIPTION_ID}'
    f'/resourceGroups/{RESOURCE_GROUP}/providers/Microsoft.App/jobs/{JOB_NAME}'
    f'?api-version={ARM_API_VERSION}'
)

token = credential.get_token('https://management.azure.com/.default').token
headers = {
    'Authorization': f'Bearer {token}',
    'Content-Type': 'application/json',
}

response = requests.put(job_url, headers=headers, json=job_payload, timeout=1800)
response.raise_for_status()
job_response = response.json()
print(json.dumps(job_response, indent=2)[:5000])

In [ ]:
verify_url = (
    f'https://management.azure.com/subscriptions/{SUBSCRIPTION_ID}'
    f'/resourceGroups/{RESOURCE_GROUP}/providers/Microsoft.App/jobs/{JOB_NAME}'
    f'?api-version={ARM_API_VERSION}'
)

verify_response = requests.get(verify_url, headers=headers, timeout=1800)
verify_response.raise_for_status()
verify_payload = verify_response.json()

print(json.dumps({
    'job_name': verify_payload['name'],
    'image': verify_payload['properties']['template']['containers'][0]['image'],
    'workload_profile_name': verify_payload['properties']['workloadProfileName'],
}, indent=2))

In [ ]:
start_url = (
    f'https://management.azure.com/subscriptions/{SUBSCRIPTION_ID}'
    f'/resourceGroups/{RESOURCE_GROUP}/providers/Microsoft.App/jobs/{JOB_NAME}/start'
    f'?api-version=2024-03-01'
)

start_response = requests.post(start_url, headers=headers, timeout=1800)
start_response.raise_for_status()
execution_response = start_response.json()
print(json.dumps(execution_response, indent=2)[:5000])

In [ ]:
resource_url = (
    f'https://portal.azure.com/#@/resource/subscriptions/{SUBSCRIPTION_ID}'
    f'/resourceGroups/{RESOURCE_GROUP}/providers/Microsoft.App/jobs/{JOB_NAME}/overview'
)

print('This job:', resource_url)
print('Cloud Shell log follow command:')
print(f'az containerapp job logs show -g {RESOURCE_GROUP} -n {JOB_NAME} --container trainer --follow --tail 50')